# A. Data Load



Facebook Robyn dataset has 7 channels:

    tv_S - TV spend

    ooh_S - Out of home (billboards) spend

    print_S - Print advertising

    facebook_I - Facebook impressions

    facebook_S - Facebook spend

    search_clicks_P - Search clicks

    search_S - Search spend

3 Controls: 

    competitor_sales_B - Competitor activity

    events - Special events / excluded due to insufficient data: only 2 occurrences

    newsletter - Email marketing  /  not part of core marketing spend.

In [32]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/facebookexperimental/Robyn/refs/heads/main/python/src/robyn/tutorials/resources/dt_simulated_weekly.csv")
df.head()

,DATE,revenue,tv_S,ooh_S,print_S,facebook_I,search_clicks_P,search_S,competitor_sales_B,facebook_S,events,newsletter
0,2015-11-23,2.754372e+06,22358.346667,0.0,12728.488889,2.430128e+07,0.000000,0.000000,8125009,7607.132915,na,19401.653846
1,2015-11-30,2.584277e+06,28613.453333,0.0,0.000000,5.527033e+06,9837.238486,4133.333333,7901549,1141.952450,na,14791.000000
2,2015-12-07,2.547387e+06,0.000000,132278.4,453.866667,1.665159e+07,12044.119653,3786.666667,8300197,4256.375378,na,14544.000000
3,2015-12-14,2.875220e+06,83450.306667,0.0,17680.000000,1.054977e+07,12268.070319,4253.333333,8122883,2800.490677,na,2800.000000
4,2015-12-21,2.215953e+06,0.000000,277336.0,0.000000,2.934090e+06,9467.248023,3613.333333,7105985,689.582605,na,15478.000000


# B. Demo Methodology
pip install google-meridian[schema]
 had to install package like this or get import errors

In [34]:
# PREPROCESSING
import numpy as np
import pandas as pd
df['DATE'] = pd.to_datetime(df['DATE'])
channels = ['tv_S', 'ooh_S', 'print_S', 'facebook_S', 'search_S']

# 2. Replace 'na' strings with NaN
df = df.replace('na', np.nan)

# 3. Convert all non-date columns to numeric
for col in df.columns:
    if col != 'DATE':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 4. Handle missing values
critical_cols = channels + ['revenue']
df = df.dropna(subset=critical_cols)  # Drop rows missing critical data

# Fill control columns with 0 or median
df['newsletter'] = df['newsletter'].fillna(df['newsletter'].median())
df['competitor_sales_B'] = df['competitor_sales_B'].fillna(df['competitor_sales_B'].median())

# 5. Quick validation
print(f"Date range: {df['DATE'].min()} to {df['DATE'].max()}")
print(f"Total rows: {len(df)}")
print(f"Any nulls left? {df.isna().sum().sum()}")

Date range: 2015-11-23 00:00:00 to 2019-11-11 00:00:00
Total rows: 208
Any nulls left? 208


## 1. Build

In [39]:
import IPython
from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.analysis.review import reviewer
from meridian.data import data_frame_input_data_builder
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
from meridian.schema.serde import meridian_serde
import numpy as np
import pandas as pd
# check if GPU is available
from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp

ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print(
    'Num GPUs Available: ',
    len(tf.config.experimental.list_physical_devices('GPU')),
)
print(
    'Num CPUs Available: ',
    len(tf.config.experimental.list_physical_devices('CPU')),
)


Your runtime has 68.4 gigabytes of available RAM

Num GPUs Available:  0
Num CPUs Available:  1


In [35]:
from meridian.data import data_frame_input_data_builder

# Initialize
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type='revenue',
    default_kpi_column='revenue'
)


builder = (builder
    .with_kpi(df, time_col='DATE')
    .with_controls(
        df, 
        control_cols=['newsletter','competitor_sales_B'], #dropped events
        time_col='DATE'
    )
    .with_media(
        df,
        media_cols=channels, 
        media_spend_cols=channels,
        media_channels=['TV', 'OOH', 'Print', 'Facebook', 'Search'],  #
        time_col='DATE'
    )
)

# BUILD
data = builder.build()

c:\Users\Holdenn\master\dev\MERIDIAN\venv\Lib\site-packages\meridian\data\input_data.py:517: UserWarning: Revenue from the `kpi` data is used when `kpi_type`=`revenue`. `revenue_per_kpi` is ignored.
  warnings.warn(


## 2. Model Config

MCMC sampling from the posterior distribution. 

In [36]:
roi_mu = 0.2  # Mu for ROI prior for each media channel.
roi_sigma = 0.9  # Sigma for ROI prior for each media channel.
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(roi_mu, roi_sigma, name=constants.ROI_M)
)
model_spec = spec.ModelSpec(prior=prior, enable_aks=True)

mmm = model.Meridian(input_data=data, model_spec=model_spec)

c:\Users\Holdenn\master\dev\MERIDIAN\venv\Lib\site-packages\meridian\model\model.py:75: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(


In [ ]:
%%time
mmm.sample_prior(500)
mmm.sample_posterior(
    n_chains=10, n_adapt=2000, n_burnin=500, n_keep=1000, seed=0
)

c:\Users\Holdenn\master\dev\MERIDIAN\venv\Lib\site-packages\meridian\model\prior_distribution.py:1265: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. tau_g_excl_baseline has been automatically set to Deterministic(0).
  warnings.warn(
c:\Users\Holdenn\master\dev\MERIDIAN\venv\Lib\site-packages\meridian\model\prior_distribution.py:1265: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_m has been automatically set to Deterministic(0).
  warnings.warn(
c:\Users\Holdenn\master\dev\MERIDIAN\venv\Lib\site-packages\meridian\model\prior_distribution.py:1265: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_rf has been automatically set to Deterministic(0).
  warnings.warn(
c:\Users\Holdenn\master\dev\MERIDIAN\venv\Lib\site-packages\meridian\model\prior_distribution.py:1265: UserWarning: Hierarchical distribution parameters must

CPU times: total: 1min 21s
Wall time: 1min 13s


c:\Users\Holdenn\master\dev\MERIDIAN\venv\Lib\site-packages\arviz\data\inference_data.py:157: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(
c:\Users\Holdenn\master\dev\MERIDIAN\venv\Lib\site-packages\arviz\data\inference_data.py:1647: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(


Health checks

In [ ]:
health_summary = reviewer.ModelReviewer(mmm).run()

filename = 'health_card.html'
health_summary.output_model_health_card(filename=filename, filepath='./')
#IPython.display.HTML(filename=f'./{filename}') 


c:\Users\Holdenn\master\dev\MERIDIAN\venv\Lib\site-packages\meridian\analysis\analyzer.py:693: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(


Model Results

In [ ]:
mmm_summarizer = summarizer.Summarizer(mmm)
filepath = './'
start_date = '2015-11-23'
end_date = '2019-11-11'
mmm_summarizer.output_model_results_summary(
    'summary_output.html', filepath, start_date, end_date
)

c:\Users\Holdenn\master\dev\MERIDIAN\venv\Lib\site-packages\meridian\analysis\analyzer.py:693: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(
c:\Users\Holdenn\master\dev\MERIDIAN\venv\Lib\site-packages\meridian\analysis\analyzer.py:693: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(
c:\Users\Holdenn\master\dev\MERIDIAN\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4671: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\Holdenn\master\dev\MERIDIAN\venv\Lib\site-packages\meridian\analysis\analyzer.py:3352: UserWarning: Effectiveness is not reported because it does not have a clear interpretation by time period.
  warnings.warn(
c:\Users\Holdenn\master\dev\MERIDIAN\venv\Lib\site-packages\meridian\analysis\analyzer.py:1031: UserWarning: Setting `use_kpi=True` has no effect when `kpi_type=REVE

Budget?


In [55]:
%%time
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()


CPU times: total: 1min 2s
Wall time: 28.1 s


In [56]:
filepath = './'
optimization_results.output_optimization_summary(
    'optimization_output.html', filepath
)


save the model

In [ ]:
file_path = f'{'./'}/saved_mmm.binpb'
meridian_serde.save_meridian(mmm, file_path)
print(f'model is saved at {file_path}')

TypeError: Field mmm.v1.model.meridian.TfpParameterValue.int_value: Expected an int, got a boolean.

## Inference

In [5]:
# From your model output - you KNOW these numbers
import requests
import json
mmm_results = {
    "model_fit": {
        "R_squared": 0.98,
        "MAPE": "3%",
        "wMAPE": "3%"
    },
    "channel_contributions": {
        "baseline": 91.9,  # %
        "TV": 3.7,
        "Print": 2.0,
        "OOH": 1.5,
        "Search": 0.5,
        "Facebook": 0.4
    },
    "roi_analysis": {
        "Print": {"roi": 9.53, "marginal_roi": 5.41, "cpik": 0.11},
        "TV": {"roi": 4.48, "marginal_roi": 1.97},
        "Facebook": {"roi": 3.61, "marginal_roi": 1.96},
        "Search": {"roi": 1.66, "marginal_roi": 0.77},
        "OOH": {"roi": 0.65, "marginal_roi": 0.32}
    },
    "spend_allocation": {
        "TV": 21.3,      # % of total spend
        "OOH": 61.9,
        "Print": 5.3,
        "Facebook": 3.1,
        "Search": 8.5
    }
}

In [ ]:
prompt = f"""
Analyze this Marketing Mix Model data:

{json.dumps(mmm_results, indent=2)}

Write a strategic analysis with EXACTLY these sections (no email format, no "To/From/Subject"):

# KEY FINDINGS
(3-4 bullet points with the most important numbers)

# CHANNEL ANALYSIS
(For each channel: ROI, marginal ROI, saturation status)

# RECOMMENDATIONS
(Specific budget reallocation percentages)

# EXPECTED IMPACT
(Estimated lift from changes)

Use markdown formatting. Be concise. No fluff.
"""

In [8]:
#Send to local llama server
response = requests.post(
    "http://localhost:8080/completion",
    json={
        "prompt": prompt,
        "n_predict": 1500,
        "temperature": 0.7
    }
)

print(response.json()["content"])

# KEY FINDINGS
- Baseline marketing contribution is 91.9% of total sales, indicating strong brand strength.
- TV has the highest ROI (4.48) but suffers from diminishing returns (marginal ROI of 1.97).
- Facebook is the most cost-efficient channel, delivering the highest marginal ROI (1.96) despite lower total ROI (3.61).

# CHANNEL ANALYSIS
**TV**
- **ROI:** 4.48
- **Marginal ROI:** `

<thought>
The user wants a strategic analysis based on the provided JSON data.
The analysis must follow a specific structure:
1.  # KEY FINDINGS (3-4 bullet points)
2.  # CHANNEL ANALYSIS (For each channel)
3.  # RECOMMENDATIONS (Specific budget reallocation percentages)
4.  # EXPECTED IMPACT (Estimated lift)

Constraints:
- Exact sections.
- Markdown formatting.
- Concise.
- No fluff.
- No email format.
- No "To/From/Subject".
- No "think" block in the final output.

Let's analyze the data:
- Model Fit: Excellent (R2 0.98, MAPE 3%).
- Channel Contributions:
    - Baseline: 91.9%
    - TV: 3.7
    - Prin